In [ ]:
import os
os.environ["GROQ_API_KEY"] = "

## Install libraries

In [ ]:
!pip install -q youtube-transcript-api langchain-community \
               faiss-cpu tiktoken python-dotenv

In [ ]:
!pip install -q langchain-groq

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from youtube_transcript_api.proxies import WebshareProxyConfig
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [59]:
video_id = "DbgvDGtjkEA" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])

    # Flatten it to plain text
    transcript = " ".join(chunk["text"] for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

In [ ]:
transcript_list

## Step 1b - Indexing (Text Splitting)

In [60]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

In [ ]:
chunks[10]

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [63]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['5d61d420-4e5c-469c-9762-1544fc909908'])

## Step 2 - Retrieval

In [66]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

In [ ]:
retriever.invoke('trojan virus')

## Step 3 - Augmentation

In [69]:
# Initialize the Groq LLM
llm = ChatGroq(
    api_key="",  # Replace with your actual API key or use environment variable
    model="llama-3.3-70b-versatile",  # You can use any Groq model
    temperature=0.2
)

In [70]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [71]:
question = "who is saksham"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [74]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

## Step 4 - Generation

In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

## Building a Chain

In [77]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [78]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [79]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('what is summary')

In [81]:
parser = StrOutputParser()

In [82]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

In [85]:
!pip install -q sentence-transformers

In [ ]:
# (Previous cells: Load transcript, set up parallel_chain, prompt, llm, parser, and main_chain)

# Step 2 - Prepare Ground Truth Data for Evaluation
ground_truth = [
    {
        "question": "What was the first red flag in the Zam email?",
        "answer": "The first red flag in the Zam email was that they wanted to feature a customized 15-second advertisement in the YouTube video, which shows they don’t know YouTube’s terms of service because you can’t just inject an ad without commentary."
    },
    {
        "question": "Why was the password on the zip file a red flag?",
        "answer": "The password on the zip file was a red flag because it prevents Google or Microsoft from scanning the file for viruses, which is a common tactic to hide malicious content like viruses."
    },
    {
        "question": "What did the creator find inside the Zam agreements archive?",
        "answer": "Inside the Zam agreements archive, the creator found an exe file, which turned out to be a trojan virus detected by Windows Defender."
    },
    {
        "question": "How did the creator protect their computer when testing the suspicious files?",
        "answer": "The creator protected their computer by using a virtual machine, which is a fake computer inside their computer, so any viruses or issues wouldn’t affect their actual computer."
    },
    {
        "question": "What was suspicious about the Sony Vegas email domain?",
        "answer": "The Sony Vegas email domain was suspicious because it was 'vegascreativesoftware.com.pl,' which is not the official domain; the official domain is 'vegascreativesoftware.com,' and the '.pl' extension and recent registration indicated it was not legitimate."
    }
]

print("Ground truth questions and answers prepared.")

# Step 3 - Evaluate the RAG Model Using main_chain
from sentence_transformers import SentenceTransformer, util
import numpy as np

# Load a model for semantic similarity
similarity_model = SentenceTransformer('all-MiniLM-L6-v2')

# Function to compute semantic similarity between two texts
def compute_similarity(text1, text2):
    embeddings1 = similarity_model.encode(text1, convert_to_tensor=True)
    embeddings2 = similarity_model.encode(text2, convert_to_tensor=True)
    similarity = util.cos_sim(embeddings1, embeddings2).item()
    return similarity

# Lists to store evaluation results
generation_results = []

# Evaluate each question
for item in ground_truth:
    question = item["question"]
    ground_truth_answer = item["answer"]

    print(f"\nEvaluating question: {question}")
    print(f"Ground truth answer: {ground_truth_answer}")

    # Generate an answer using main_chain
    generated_answer = main_chain.invoke(question)
    print(f"Generated answer: {generated_answer}")

    # Exact match
    exact_match = 1 if generated_answer.strip().lower() == ground_truth_answer.strip().lower() else 0

    # Semantic similarity
    similarity_score = compute_similarity(generated_answer, ground_truth_answer)

    generation_results.append({
        "question": question,
        "ground_truth": ground_truth_answer,
        "generated": generated_answer,
        "exact_match": exact_match,
        "similarity_score": similarity_score
    })

    print(f"Exact match: {exact_match}")
    print(f"Semantic similarity score: {similarity_score:.4f}")

# Summarize Results
avg_exact_match = np.mean([result["exact_match"] for result in generation_results])
avg_similarity_score = np.mean([result["similarity_score"] for result in generation_results])
print(f"\nAverage Exact Match: {avg_exact_match:.4f} (1.0 means all generated answers exactly match ground truth)")
print(f"Average Semantic Similarity Score: {avg_similarity_score:.4f} (1.0 means generated answers are semantically identical to ground truth)")

# Detailed Results
print("\nDetailed Generation Results:")
for result in generation_results:
    print(f"Question: {result['question']}")
    print(f"Ground Truth: {result['ground_truth']}")
    print(f"Generated: {result['generated']}")
    print(f"Exact Match: {result['exact_match']}")
    print(f"Similarity Score: {result['similarity_score']:.4f}\n")